# ATLAS — Multimodal Training

Boilerplate for feature-level late fusion. **All inputs are cleaned, preprocessed, and null-handled.** Fill in the TODO cells.

| Component | Owner | Output |
| --- | --- | --- |
| S3 image tensors → ResNet-18 | Kevin | `[B, 512]` |
| Tabular features → MLP | Swati | `[B, 64]` |
| USGS DEM → SmallCNN | Abhi | `[B, 64]` |
| Fusion, training, evaluation, XAI | Team — owner TBD | `[B, 640]` → logits |

`B` = batch size. Concatenation order: **image, tabular, DEM**.

## 1. Setup
**Owner: Team — TBD**

Set imports, device, seed, S3 paths, class names, and training hyperparameters.

In [ ]:
# TODO: Imports, device, seed, paths, classes, and hyperparameters.

## 2. Load aligned inputs and split assignments
**Owner: Team — TBD**

Load the prepared facility manifest and shared train/validation/test assignments. Use facility IDs to retrieve matching image, tabular, DEM, and target inputs.

In [ ]:
# TODO: Load the prepared manifest and shared split assignments.

## 3. Image encoder — Kevin

Load prepared image tensors from S3 (`satellite/processed/`). Each `.pt` contains `id`, `image`, and `source_key`.

**`[B, 3, 224, 224]` → ResNet-18 → `[B, 512]`**

Extract features before the final classification layer. Specify checkpoint and frozen/trainable layers.

In [ ]:
# TODO (Kevin): Load S3 image tensors and define the ResNet-18 feature encoder.
# [B, 3, 224, 224] -> [B, 512]

## 4. Tabular encoder — Swati

Load prepared IM3 + EIA + WRI features and adapt `TabularEncoder` for fusion.

**`[B, F]` → MLP → `[B, 64]`**

`F` is the prepared feature count. Return embeddings; train with the fused classification loss.

In [ ]:
# TODO (Swati): Load prepared tabular features and define the MLP encoder.
# [B, F] -> [B, 64]

In [ ]:
# Tabular config + load 
import io
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

TAB_BUCKET = "atlas-acm-research-f26"
TAB_KEY = "dataset/im3_atlas_wri_13scorescombined_normalized_0to1.csv"  
TAB_PROFILE = "ATLAS"
TAB_EXPECTED_ROWS = 317                                   
TAB_EMB_DIM = 64

# shared train/val/test locally on my computer 
# Section 2 should replace to shared location for split assignments.
TAB_SPLITS_DIR = Path("/Users/swati/ATLAS/Data/pipeline/splits")

# never fed to the model: identifiers, text, location, and LEAKAGE columns
TAB_ID_COLS = ["id", "name", "ref", "operator", "county", "state", "state_abb", "state_id", "county_id",
               "county_fips", "pfaf_id", "retail_period", "period"]
TAB_LOCATION_COLS = ["lat", "lon"]          # so the model can't just memorize location
TAB_LEAKAGE_COLS = ["sqft", "size_class"]   # size_class is built from sqft -> leaks the size answer
TAB_DROP_COLS = TAB_ID_COLS + TAB_LOCATION_COLS + TAB_LEAKAGE_COLS

tab_s3 = boto3.Session(profile_name=TAB_PROFILE, region_name="us-east-2").client("s3")
body = tab_s3.get_object(Bucket=TAB_BUCKET, Key=TAB_KEY)["Body"].read()
tab_df = pd.read_csv(io.BytesIO(body), dtype={"id": str, "state_id": str, "county_id": str})
tab_df["id"] = tab_df["id"].str.strip().str.zfill(10)      # restore leading zeros if they were lost
tab_df = tab_df.reset_index(drop=True)

assert len(tab_df) == TAB_EXPECTED_ROWS, (
    f"{TAB_KEY} has {len(tab_df)} rows, expected {TAB_EXPECTED_ROWS}. "
    "It's probably still built on the old 1147 baseline; use Abhi's updated EIA+WRI file.")
assert tab_df["id"].is_unique, "duplicate facility ids"
print("tabular file:", tab_df.shape)

In [ ]:
# Split assignment (same facilities in train/val/test as the pipeline)
tab_split_of = {}
for name in ["train", "val", "test"]:
    f = TAB_SPLITS_DIR / f"{name}.csv"
    assert f.exists(), f"Missing {f}. Run eia_feature_pipeline.ipynb first, or fix TAB_SPLITS_DIR."
    ids = pd.read_csv(f, usecols=["id"], dtype={"id": str})["id"].str.zfill(10)
    tab_split_of.update({i: name for i in ids})

tab_df["split"] = tab_df["id"].map(tab_split_of)
missing = tab_df.loc[tab_df["split"].isna(), "id"].tolist()
assert not missing, f"{len(missing)} ids aren't in the pipeline splits, e.g. {missing[:5]}"
assert len(tab_split_of) == len(tab_df), f"splits have {len(tab_split_of)} ids, file has {len(tab_df)}"

tab_train_ids = tab_df.loc[tab_df["split"] == "train", "id"].tolist()
tab_val_ids = tab_df.loc[tab_df["split"] == "val", "id"].tolist()
tab_test_ids = tab_df.loc[tab_df["split"] == "test", "id"].tolist()
print("train / val / test:", len(tab_train_ids), len(tab_val_ids), len(tab_test_ids))

In [ ]:
# Pick features + preprocess (impute/scale numeric, one-hot categorical). FIT ON TRAIN ONLY.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

tab_feat_df = tab_df.drop(columns=[c for c in TAB_DROP_COLS + ["split"] if c in tab_df.columns])
train_mask = (tab_df["split"] == "train").values

# drop columns that are constant in train (no information)
tab_constant = [c for c in tab_feat_df.columns if tab_feat_df.loc[train_mask, c].nunique(dropna=False) <= 1]
tab_feat_df = tab_feat_df.drop(columns=tab_constant)

tab_num_cols = tab_feat_df.select_dtypes(include=np.number).columns.tolist()
tab_cat_cols = [c for c in tab_feat_df.columns if c not in tab_num_cols]
for c in TAB_LEAKAGE_COLS + TAB_LOCATION_COLS:
    assert c not in tab_feat_df.columns, f"{c} is still a feature"

tab_preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), tab_num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), tab_cat_cols),
])
tab_preprocess.fit(tab_feat_df.loc[train_mask])                      # train rows only
tab_X = tab_preprocess.transform(tab_feat_df).astype(np.float32)     # applied to every row
assert not np.isnan(tab_X).any(), "NaNs left after preprocessing"

TAB_FEATURE_NAMES = list(tab_preprocess.get_feature_names_out())
TAB_IN_DIM = tab_X.shape[1]                                          # F, never hardcoded

# id -> row lookup, used by the Dataset in section 6
tab_row_of = {i: r for r, i in enumerate(tab_df["id"])}

def get_tabular(ids):
    """Tabular feature tensor [B, F] for a list of facility ids (same order as ids)."""
    return torch.from_numpy(tab_X[[tab_row_of[i] for i in ids]])

print("Dropped (ids/location/leakage):", [c for c in TAB_DROP_COLS if c in tab_df.columns])
print("Dropped constant:", tab_constant)
print(f"Numeric ({len(tab_num_cols)}):", tab_num_cols)
print(f"Categorical ({len(tab_cat_cols)}):", tab_cat_cols)
print("F =", TAB_IN_DIM)

In [ ]:
# 4d. Tabular MLP encoder: [B, F] -> [B, 64]. Returns embeddings only (no head);
class TabularEncoder(nn.Module):
    def __init__(self, in_dim, emb_dim=TAB_EMB_DIM, hidden=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, emb_dim), nn.LayerNorm(emb_dim),
        )
    def forward(self, x):
        return self.net(x)

# shape check on a few training facilities
_enc = TabularEncoder(TAB_IN_DIM).eval()
with torch.no_grad():
    _out = _enc(get_tabular(tab_train_ids[:8]))
assert _out.shape == (8, TAB_EMB_DIM), _out.shape
print(f"TabularEncoder OK: [8, {TAB_IN_DIM}] -> {list(_out.shape)}")
del _enc, _out

## 5. DEM encoder — Abhi

Load prepared USGS DEM tensors and implement SmallCNN.

**`[B, C_dem, H_dem, W_dem]` → SmallCNN → `[B, 64]`**

Return embeddings before any branch classification head.

In [ ]:
# TODO (Abhi): Load prepared USGS DEM tensors and define SmallCNN.
# [B, C_dem, H_dem, W_dem] -> [B, 64]

## 6. Dataset and DataLoaders
**Owner: Team — TBD**

Each sample returns image, tabular, DEM, label, and facility ID. Build train, validation, and test loaders with matching modality order.

In [ ]:
# TODO: Multimodal Dataset and train/validation/test DataLoaders.

## 7. Feature concatenation and classifier
**Owner: Team — TBD**

Concatenate along `dim=1`, in image/tabular/DEM order:

**`[B, 512] + [B, 64] + [B, 64] → [B, 640] → classifier → [B, K]`**

`K` = number of classes. Return logits for cross-entropy loss.

In [ ]:
# TODO: Concatenate image, tabular, DEM embeddings and define the classifier.
# [B, 640] -> [B, K] logits

## 8. Integration check
**Owner: Team — TBD**

Run one batch. Verify embedding/logit shapes, compute loss, and confirm gradients reach all intended trainable components.

In [ ]:
# TODO: One-batch forward pass, loss, and gradient checks.

## 9. Training
**Owner: Team — TBD**

Define loss, optimizer, scheduler, and epochs. Track train/validation loss and macro-F1; use validation for early stopping and best-checkpoint selection. Save weights and model configuration.

In [ ]:
# TODO: Training, validation, early stopping, and best-checkpoint saving.

## 10. Evaluation
**Owner: Team — TBD**

Evaluate the best checkpoint on held-out test facilities. Report accuracy, macro-F1, per-class precision/recall/F1, and a confusion matrix. Export predictions.

Compare on the same test facilities: image only, image + tabular, image + DEM, and all three modalities.

In [ ]:
# TODO: Test metrics, confusion matrix, predictions, and modality ablations.

## 11. XAI scope — proposed
**Owner: Team — TBD**

| Component | Method |
| --- | --- |
| Image | Grad-CAM on ResNet-18 |
| Tabular | Integrated Gradients or SHAP |
| DEM | Input attribution or SmallCNN Grad-CAM |
| Modalities | Ablation comparison |

Target the final fused prediction. Start with image Grad-CAM and modality ablations; add tabular/DEM explanations afterward. Show a few correct and incorrect test predictions.

In [ ]:
# TODO: XAI methods and example visualizations.

## 12. Results and handoff
**Owner: Team — TBD**

Record checkpoint, metrics, ablations, XAI examples, artifact paths, and remaining tasks by owner.

In [ ]:
# TODO: Experiment summary and artifact paths.